In [1]:
import tensorflow as tf
import numpy as np

In [2]:
image = tf.random.uniform(shape = (224,224,3), minval = 8, maxval=255, dtype=tf.float32)
single_image_batch = tf.expand_dims(image, axis = 0)
print("original Image Shape:", single_image_batch.shape)

original Image Shape: (1, 224, 224, 3)


In [4]:
patch_size = 16
patches = tf.image.extract_patches(
    images  = single_image_batch,
    sizes = [1, patch_size, patch_size,1],
    strides = [1, patch_size, patch_size,1],
    rates = [1,1,1,1],
    padding = "VALID"
)
print("extraxted grid shape", patches.shape)

extraxted grid shape (1, 14, 14, 768)


In [11]:
new_patches = patches.shape[1]*patches.shape[2]
patch_dim = patches.shape[-1]

patches_sequences = tf.reshape(patches, (1, new_patches, patch_dim))
print("patches sequences shape", patches_sequences.shape)

patches sequences shape (1, 196, 768)


In [13]:
embed_dim = 512
projection_layer = tf.keras.layers.Dense(embed_dim)
embedded_patches = projection_layer(patches_sequences)
print("final Embeddings Shape:", embedded_patches.shape)

final Embeddings Shape: (1, 196, 512)


In [14]:
print(patches)

tf.Tensor(
[[[[ 93.48372  154.40738   40.08023  ...   8.277369  84.49587
    163.00087 ]
   [ 87.10661  179.05748  224.50363  ... 239.2003   117.984764
     82.814285]
   [ 48.28994  109.08437   69.84802  ... 141.97404   91.71739
    163.78784 ]
   ...
   [166.95863  225.48196   95.285614 ... 119.11765  136.96158
     40.70696 ]
   [ 21.650766 199.25577   39.776863 ... 225.53831   25.645735
     40.41375 ]
   [ 65.05437   31.900288 226.41333  ... 230.10548  150.91685
     45.918526]]

  [[100.95823  208.79315  246.2016   ... 110.353966 237.6534
    193.41356 ]
   [151.85359   36.254158 148.13466  ... 198.08157  160.73071
     39.55255 ]
   [ 88.482185  31.297909 249.18211  ...  86.16318   66.6788
    240.90373 ]
   ...
   [ 88.747955 244.43147  233.32095  ...  43.711823 148.69977
    104.34027 ]
   [175.05247  216.8241   112.695175 ...  25.578306  29.00882
     98.180855]
   [ 69.20198  190.64001  240.33998  ...  64.14906  252.60918
     93.01761 ]]

  [[ 35.263252 130.75885  107.58681

In [16]:
from tensorflow.keras import layers

class ViTPatchAndPositionEmbedding(layers.Layer):
  def __init__(self, num_patches = 196, embed_dim = 512, **kwargs):
    super(ViTPatchAndPositionEmbedding, self).__init__(**kwargs)
    self.num_patches = num_patches
    self.embed_dim = embed_dim

    self.position_embedding = layers.Embedding(
        input_dim = num_patches,
        output_dim = embed_dim
    )

  def call(self, patch_embeddings):

    positions = tf.range(start = 0, limit = self.num_patches, delta = 1)
    pos_embeddings = self.position_embedding(positions)
    return patch_embeddings + pos_embeddings

In [17]:
dummy_patch_embeddings = tf.random.normal((1, 196, 512))

pos_layer = ViTPatchAndPositionEmbedding(num_patches=196, embed_dim=512)
final_vit_input = pos_layer(dummy_patch_embeddings)

print("Patch Embeddings Shape:", dummy_patch_embeddings.shape)
print("Final Output Shape:    ", final_vit_input.shape)

Patch Embeddings Shape: (1, 196, 512)
Final Output Shape:     (1, 196, 512)
